# Voice Symptom Intake & Documentation Assistant - Google Colab Deployment

This notebook deploys the Voice Symptom Intake & Documentation Assistant on Google Colab with GPU support.

## ✅ Advantages of Colab Deployment:
- **Free GPU Access** (Tesla T4 with 16GB VRAM)
- **No Local Setup Issues** (no Windows file locking, FFmpeg, etc.)
- **Faster Inference** (MedASR & MedGemma both run on GPU)
- **Public URL Access** via ngrok

## ⚠️ Important Notes:
- Sessions last up to 12 hours (free tier)
- You'll need a **Hugging Face token** with "Read" access
- Accept model terms for `google/medasr` and `google/medgemma-1.5-4b-it`

## 🆕 Latest Updates:
- **Medical NER** — Automatic extraction of Conditions & Medications using SciSpaCy
- **Progressive Web App** — Offline recording, install-to-homescreen, background sync
- **Real-time streaming transcription** via WebSocket (words appear as you speak!)
- Enhanced JSON parsing for reliable documentation generation
- Robust error recovery with intelligent fallback strategies
- Optimized generation parameters for medical documentation

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

%%capture
# Install transformers from specific commit for MedASR support
!pip install git+https://github.com/huggingface/transformers.git@65dc261512cbdb1ee72b88ae5b222f2605aad8e5

# Install other dependencies
!pip install fastapi uvicorn[standard] python-multipart websockets
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install accelerate librosa soundfile noisereduce audioread
!pip install pydantic pydantic-settings python-dotenv
!pip install pyngrok nest-asyncio

# Medical NER (SciSpaCy)
!pip install spacy scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

print("✅ All dependencies installed successfully!")

In [ ]:
%%capture
# Install transformers from specific commit for MedASR support
!pip install git+https://github.com/huggingface/transformers.git@65dc261512cbdb1ee72b88ae5b222f2605aad8e5

# Install other dependencies
!pip install fastapi uvicorn[standard] python-multipart websockets
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install accelerate librosa soundfile noisereduce audioread
!pip install pydantic pydantic-settings python-dotenv
!pip install pyngrok nest-asyncio openai-whisper

print("✅ All dependencies installed successfully!")

## Step 3: Upload Your Code

**Option A:** Upload project folder from your computer
- Click the folder icon on the left sidebar
- Upload the entire `voice-symptom-triage-assistant` folder

**Option B:** Clone from GitHub (if you've pushed your code)

In [ ]:
# Option B: Clone from GitHub (uncomment and modify if using)
# !git clone https://github.com/YOUR_USERNAME/voice-symptom-triage-assistant.git
# %cd voice-symptom-triage-assistant

# Option A: If uploaded manually, navigate to the folder
%cd /content/voice-symptom-triage-assistant

## Step 4: Configure Hugging Face Token & Environment

**Get your token from:** https://huggingface.co/settings/tokens

**Make sure you've accepted terms for:**
- https://huggingface.co/google/medasr
- https://huggingface.co/google/medgemma-1.5-4b-it

In [ ]:
import os

# REPLACE WITH YOUR HUGGING FACE TOKEN
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

# Create .env file with enhanced MedGemma parameters
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write("MEDASR_MODEL=google/medasr\n")
    f.write("MEDGEMMA_MODEL=google/medgemma-1.5-4b-it\n")
    f.write("DEVICE=cuda\n")
    f.write("ENABLE_GPU=true\n")
    
    # MedGemma Generation Parameters (optimized for JSON output)
    f.write("MEDGEMMA_MAX_TOKENS=1024\n")
    f.write("MEDGEMMA_REPETITION_PENALTY=1.1\n")
    
    # Audio settings
    f.write("AUDIO_SAMPLE_RATE=16000\n")
    f.write("MAX_AUDIO_DURATION_SECONDS=300\n")
    
    # Streaming transcription (GPU can handle 2s intervals)
    f.write("STREAMING_INTERVAL_SECONDS=2.0\n")
    
    # Model cache directory
    f.write("MODEL_CACHE_DIR=./models\n")
    f.write("LOG_LEVEL=INFO\n")

print("✅ Environment configured with optimized parameters!")
print("   - Max Tokens: 1024 (complete documentation)")
print("   - Repetition Penalty: 1.1 (prevent loops)")
print("   - Streaming Interval: 2.0s (real-time transcription)")
print("   - Using greedy decoding for stable GPU inference")

import subprocess
import threading
import time

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🌐 PUBLIC URL: {public_url}")
print(f"{'='*60}\n")
print("Open this URL in your browser to access the application!\n")
print("🆕 Features in this deployment:")
print("   ✓ Real-time streaming transcription via WebSocket")
print("   ✓ Live word-by-word transcript display")
print("   ✓ Medical NER — Condition & Medication extraction (SciSpaCy)")
print("   ✓ Progressive Web App — Offline recording & background sync")
print("   ✓ Enhanced JSON parsing for reliable documentation")
print("   ✓ Intelligent fallback if JSON parsing fails")
print("   ✓ Optimized generation parameters")
print("   ✓ GPU acceleration for faster results\n")

# Start uvicorn server
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

In [ ]:
## Step 8: Access Your Application

1. **Copy the public ngrok URL** from the output above
2. **Open it in your browser**
3. **Start recording** — words will appear in real-time as you speak!
4. **Click 'Generate Documentation'** to get structured SOAP notes

## ✅ Features:
- **🎙️ Real-time streaming transcription** (words appear as you speak!)
- **🧬 Medical NER** — auto-extracts Conditions, Medications with ICD-10/RxNorm codes
- **📱 Progressive Web App** — works offline, install to homescreen on tablets
- Audio recording directly in browser
- File upload (WAV, MP3, M4A, FLAC, OGG)
- Real-time transcription with MedASR
- Structured documentation with MedGemma
- **Enhanced JSON parsing** - no more "N/A" fields!
- **Robust error recovery** - always generates documentation
- Export results as JSON or PDF
- Copy to clipboard

## 🛑 To Stop:
- Click the **Stop** button in Colab
- Or press **Ctrl+C** in the cell output

## 📝 Notes:
- Free Colab sessions last **up to 12 hours**
- The ngrok URL **changes each time** you restart
- Models are **cached** after first download (faster restarts)
- GPU inference is **10x faster** than CPU
- WebSocket streaming works through ngrok tunnels automatically
- Documentation fields should now populate correctly (check logs for parsing_method)

## Troubleshooting

### If models fail to load:
1. Check your HF_TOKEN is valid
2. Verify you accepted model terms on Hugging Face
3. Try restarting the runtime: Runtime → Restart runtime

### If ngrok fails:
1. Verify your ngrok authtoken
2. Free ngrok accounts have limits (1 tunnel at a time)
3. Try getting a new authtoken from ngrok dashboard

### If audio fails:
1. Check browser microphone permissions
2. Hard refresh browser (Ctrl+Shift+R)
3. Ensure audio files are under 5 minutes (default limit)

### If WebSocket streaming doesn't work:
1. Check browser console (F12) for WebSocket connection errors
2. ngrok tunnels support WebSocket natively — no extra config needed
3. Ensure FFmpeg is available: `!apt-get install -y ffmpeg` (pre-installed on Colab)
4. If streaming fails, the app automatically falls back to upload-based transcription

### If documentation shows "N/A" fields:
1. Check the server logs for "parsing_method" messages
2. Look for "json_successful" or "text_extraction_fallback"
3. If seeing frequent fallback, the model may need more warmup time
4. Check logs for any JSON extraction warnings

### If Medical NER shows no entities:
1. Check that SciSpaCy models installed correctly: `import en_ner_bc5cdr_md`
2. Ensure the transcript contains medical terms (conditions or medications)
3. Check server logs for `Medical NER ready: True`
4. Try restarting the runtime if models failed to load

### Debug Mode:
If you want to see detailed MedGemma output for debugging:
```python
# In a new cell
import logging
logging.getLogger('app.models.medgemma_service').setLevel(logging.DEBUG)
```
Then restart the server to see detailed parsing logs.

In [ ]:
from pyngrok import ngrok
import nest_asyncio

# REPLACE WITH YOUR NGROK AUTHTOKEN
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"

# Set up ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
nest_asyncio.apply()

print("✅ ngrok configured!")

## Step 7: Start the Server

This will:
1. Load MedASR model on GPU (~30 seconds)
2. Load MedGemma model on GPU (~2-3 minutes first time)
3. Start the FastAPI server with WebSocket support
4. Create a public ngrok URL

**Note:** Real-time streaming transcription and WebSocket support are included automatically!

In [ ]:
import subprocess
import threading
import time

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🌐 PUBLIC URL: {public_url}")
print(f"{'='*60}\n")
print("Open this URL in your browser to access the application!\n")
print("🆕 Features in this deployment:")
print("   ✓ Real-time streaming transcription via WebSocket")
print("   ✓ Live word-by-word transcript display")
print("   ✓ Enhanced JSON parsing for reliable documentation")
print("   ✓ Intelligent fallback if JSON parsing fails")
print("   ✓ Optimized generation parameters")
print("   ✓ GPU acceleration for faster results\n")

# Start uvicorn server
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

## Step 8: Access Your Application

1. **Copy the public ngrok URL** from the output above
2. **Open it in your browser**
3. **Start recording** — words will appear in real-time as you speak!
4. **Click 'Generate Documentation'** to get structured SOAP notes

## ✅ Features:
- **🎙️ Real-time streaming transcription** (words appear as you speak!)
- Audio recording directly in browser
- File upload (WAV, MP3, M4A, FLAC, OGG)
- Real-time transcription with MedASR
- Structured documentation with MedGemma
- **Enhanced JSON parsing** - no more "N/A" fields!
- **Robust error recovery** - always generates documentation
- Export results as JSON
- Copy to clipboard

## 🛑 To Stop:
- Click the **Stop** button in Colab
- Or press **Ctrl+C** in the cell output

## 📝 Notes:
- Free Colab sessions last **up to 12 hours**
- The ngrok URL **changes each time** you restart
- Models are **cached** after first download (faster restarts)
- GPU inference is **10x faster** than CPU
- WebSocket streaming works through ngrok tunnels automatically
- Documentation fields should now populate correctly (check logs for parsing_method)

## Troubleshooting

### If models fail to load:
1. Check your HF_TOKEN is valid
2. Verify you accepted model terms on Hugging Face
3. Try restarting the runtime: Runtime → Restart runtime

### If ngrok fails:
1. Verify your ngrok authtoken
2. Free ngrok accounts have limits (1 tunnel at a time)
3. Try getting a new authtoken from ngrok dashboard

### If audio fails:
1. Check browser microphone permissions
2. Hard refresh browser (Ctrl+Shift+R)
3. Ensure audio files are under 5 minutes (default limit)

### If WebSocket streaming doesn't work:
1. Check browser console (F12) for WebSocket connection errors
2. ngrok tunnels support WebSocket natively — no extra config needed
3. Ensure FFmpeg is available: `!apt-get install -y ffmpeg` (pre-installed on Colab)
4. If streaming fails, the app automatically falls back to upload-based transcription

### If documentation shows "N/A" fields:
1. Check the server logs for "parsing_method" messages
2. Look for "json_successful" or "text_extraction_fallback"
3. If seeing frequent fallback, the model may need more warmup time
4. Check logs for any JSON extraction warnings

### Debug Mode:
If you want to see detailed MedGemma output for debugging:
```python
# In a new cell
import logging
logging.getLogger('app.models.medgemma_service').setLevel(logging.DEBUG)
```
Then restart the server to see detailed parsing logs.